# PBIT Model Training

6 experimental configurations × 3 models, grouped 5-fold CV by `Serial_Number`.

| # | Configuration | Augmentation | Weights |
|---|---------------|:---:|---|
| 1 | Baseline | no | none |
| 2 | Freq weights | no | frequency |
| 3 | Sev weights | no | severity |
| 4 | Freq×Sev weights | no | freq × sev |
| 5 | Augmentation | yes | none |
| 6 | Aug + Freq×Sev | yes | freq × sev |

**Models**: LightGBM · XGBoost · Random Forest

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pbit.modeling import (
    EXPERIMENTS, META_COLS, FAULT_ORDER,
    XGBWrapper,
    make_cv_folds, fold_diagnostics,
    run_cv_experiment, summarize_cv, results_table,
)
from pbit.data_validation import load_long_dataset

TRAIN_REAL_PATH = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_CLEAN_PATH = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
RESULTS_PATH    = PROJECT_ROOT / 'data' / 'cv_results.csv'

PROJECT_ROOT

## Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_REAL_PATH)
long_df  = load_long_dataset(LONG_CLEAN_PATH)

feat_cols = [c for c in train_df.columns if c not in META_COLS]

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Long dataset : {len(long_df):,} rows, {long_df["Experiment_Index"].nunique():,} experiments')
train_df['Fault_Mode'].value_counts().reindex(FAULT_ORDER)

## Cross-Validation Folds

In [ ]:
folds = make_cv_folds(train_df, n_splits=5, random_state=42)
fold_diagnostics(train_df, folds)

## Define Models

In [ ]:
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

MODELS = {
    'LightGBM': (
        lgb.LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=5,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose=-1,
        ),
        False,
    ),
    'XGBoost': (
        XGBWrapper(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='mlogloss',
            random_state=42,
            verbosity=0,
        ),
        False,
    ),
    'RandomForest': (
        RandomForestClassifier(
            n_estimators=400,
            max_features='sqrt',
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
        ),
        True,
    ),
}

print('Models:', list(MODELS.keys()))
print('Configs:', [c.name for c in EXPERIMENTS])

## Run All Experiments

6 configs x 3 models = 18 runs. Each run: 5 CV folds.

In [ ]:
import time

all_results = {}
all_summaries = {}

for model_name, (model, needs_imp) in MODELS.items():
    all_results[model_name] = {}
    for cfg in EXPERIMENTS:
        label = f'{model_name} / {cfg.name}'
        t0 = time.time()
        fold_res = run_cv_experiment(
            model, cfg, train_df, long_df, feat_cols, folds,
            needs_imputation=needs_imp,
            aug_target=160, aug_noise=0.01, random_state=42,
        )
        elapsed = time.time() - t0
        all_results[model_name][cfg.name] = fold_res
        summary = summarize_cv(fold_res)
        all_summaries[label] = summary
        f1  = summary.get('macro_f1_mean', float('nan'))
        f06 = summary.get('f06_recall_mean', float('nan'))
        f05 = summary.get('f05_recall_mean', float('nan'))
        print(f'{label:<40}  F1={f1:.3f}  F06={f06:.3f}  F05={f05:.3f}  ({elapsed:.0f}s)')

## Results Table

In [ ]:
tbl = results_table(all_summaries)
tbl

## Per-Class Recall Heatmap

In [ ]:
rows = []
for label, s in all_summaries.items():
    row = {'Experiment': label}
    for cls in FAULT_ORDER:
        row[cls] = round(s['per_class_recall_mean'].get(cls, float('nan')), 3)
    rows.append(row)

heatmap_df = pd.DataFrame(rows).set_index('Experiment')

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    heatmap_df.astype(float), annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.4, ax=ax,
)
ax.set_title('Mean recall per class (5-fold CV)', fontsize=13)
ax.set_xlabel('Fault Mode')
plt.tight_layout()
plt.show()

## Severity-Level Recall

In [ ]:
sev_rows = []
for label, s in all_summaries.items():
    sev_rows.append({
        'Experiment': label,
        'Sev0 (N01,N02)': round(s.get('sev0_recall_mean', float('nan')), 3),
        'Sev1 (F01)':     round(s.get('sev1_recall_mean', float('nan')), 3),
        'Sev2 (F02,F04,F05)': round(s.get('sev2_recall_mean', float('nan')), 3),
        'Sev3 (F03,F06)': round(s.get('sev3_recall_mean', float('nan')), 3),
    })
sev_tbl = pd.DataFrame(sev_rows).set_index('Experiment')
sev_tbl

## Save Results

In [ ]:
tbl.to_csv(RESULTS_PATH, index=False)
print(f'Saved: {RESULTS_PATH}')
tbl

---
## F05 Investigation

F05 recall is near zero across all configurations. This section diagnoses why:
1. Where do F05 samples get classified? (confusion breakdown)
2. Which features separate F05 from the absorbing class?
3. Are the raw signals distinguishable?

### 1. Aggregated Confusion Matrix - LightGBM / Augmentation

In [ ]:
lgb_aug_folds = all_results['LightGBM']['Augmentation']

cm_labels = lgb_aug_folds[0]['cm_labels']
cm_agg = np.zeros((len(cm_labels), len(cm_labels)), dtype=int)
for fold in lgb_aug_folds:
    fl  = fold['cm_labels']
    fcm = fold['confusion_matrix']
    for ri, rl in enumerate(fl):
        for ci, cl in enumerate(fl):
            if rl in cm_labels and cl in cm_labels:
                r = cm_labels.index(rl)
                c = cm_labels.index(cl)
                cm_agg[r, c] += fcm[ri, ci]

cm_df = pd.DataFrame(cm_agg, index=cm_labels, columns=cm_labels)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', linewidths=0.4, ax=ax)
ax.set_title('LightGBM / Augmentation - Aggregated confusion (5 folds)', fontsize=12)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()

if 'F05' in cm_labels:
    f05_row = cm_df.loc['F05']
    print('\nF05 row (where F05 samples land):')
    print(f05_row[f05_row > 0].to_string())

### 2. F05 Absorption Across All Models and Configs

In [ ]:
f05_rows = []
for model_name in MODELS:
    for cfg in EXPERIMENTS:
        fold_res = all_results[model_name][cfg.name]
        ref_labels = fold_res[0]['cm_labels']
        cm_sum = np.zeros((len(ref_labels), len(ref_labels)), dtype=int)
        for fold in fold_res:
            fl  = fold['cm_labels']
            fcm = fold['confusion_matrix']
            for ri, rl in enumerate(fl):
                for ci, cl in enumerate(fl):
                    if rl in ref_labels and cl in ref_labels:
                        r = ref_labels.index(rl)
                        c = ref_labels.index(cl)
                        cm_sum[r, c] += fcm[ri, ci]
        cm_d = pd.DataFrame(cm_sum, index=ref_labels, columns=ref_labels)
        if 'F05' in ref_labels:
            f05_total   = int(cm_d.loc['F05'].sum())
            f05_correct = int(cm_d.loc['F05', 'F05']) if 'F05' in cm_d.columns else 0
            f05_recall  = f05_correct / f05_total if f05_total > 0 else float('nan')
            f05_wrong   = cm_d.loc['F05'].drop('F05', errors='ignore')
            top_pred   = f05_wrong.idxmax() if f05_wrong.sum() > 0 else 'none'
            top_pred_n = int(f05_wrong.max()) if f05_wrong.sum() > 0 else 0
            f05_rows.append({
                'Model': model_name, 'Config': cfg.name,
                'F05_total': f05_total, 'F05_correct': f05_correct,
                'F05_recall': round(f05_recall, 3),
                'Top_absorber': top_pred, 'Top_absorber_n': top_pred_n,
            })

f05_summary = pd.DataFrame(f05_rows)
f05_summary

### 3. Feature Separation: F05 vs Primary Absorbing Class

In [ ]:
top_absorber = f05_summary['Top_absorber'].mode()[0]
print(f'Primary absorbing class: {top_absorber}')

f05_mask = train_df['Fault_Mode'] == 'F05'
abs_mask  = train_df['Fault_Mode'] == top_absorber

f05_means = train_df.loc[f05_mask, feat_cols].mean()
abs_means = train_df.loc[abs_mask, feat_cols].mean()
f05_std   = train_df.loc[f05_mask, feat_cols].std()
abs_std   = train_df.loc[abs_mask, feat_cols].std()
pooled    = np.sqrt((f05_std**2 + abs_std**2) / 2).replace(0, np.nan)
std_diff  = ((f05_means - abs_means) / pooled).abs().dropna()

top20 = std_diff.nlargest(20)
fig, ax = plt.subplots(figsize=(8, 6))
top20[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_title(f'Top 20 features: |Cohen d| F05 vs {top_absorber}', fontsize=12)
ax.set_xlabel("|Cohen's d|")
ax.axvline(0.5, color='gray', linestyle='--', linewidth=0.8, label='d=0.5')
ax.axvline(0.8, color='orange', linestyle='--', linewidth=0.8, label='d=0.8')
ax.legend()
plt.tight_layout()
plt.show()

print(f'\nTop 20 (F05 vs {top_absorber}):')
print(top20.to_string())

### 4. Signal Envelopes: F05, F04, Absorbing Class

In [ ]:
f05_ids = train_df.loc[train_df['Fault_Mode'] == 'F05', 'Experiment_Index'].tolist()
f04_ids = train_df.loc[train_df['Fault_Mode'] == 'F04', 'Experiment_Index'].tolist()
abs_ids = train_df.loc[train_df['Fault_Mode'] == top_absorber, 'Experiment_Index'].tolist()

def signal_envelope(long_df, exp_ids):
    sub = long_df[long_df['Experiment_Index'].isin(exp_ids)]
    g = sub.groupby('Sample_Index')['Measurement']
    return g.mean(), g.std()

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
groups = [
    (f05_ids, 'F05',  'tomato'),
    (f04_ids, 'F04',  'steelblue'),
    (abs_ids, top_absorber, 'seagreen'),
]
for ax, (ids, name, color) in zip(axes, groups):
    mu, sigma = signal_envelope(long_df, ids)
    x = mu.index
    ax.plot(x, mu.values, color=color, linewidth=2, label=f'{name} mean (n={len(ids)})')
    ax.fill_between(x, (mu - sigma).values, (mu + sigma).values,
                    color=color, alpha=0.25, label='+-1 std')
    ax.set_ylabel('Measurement')
    ax.legend(loc='upper right')
    ax.grid(True, linewidth=0.3)
axes[-1].set_xlabel('Sample Index')
fig.suptitle('Signal envelopes (training set real experiments)', fontsize=13)
plt.tight_layout()
plt.show()

### 5. Feature Distributions: Top Discriminative Features

In [ ]:
top6_feats = top20.head(6).index.tolist()
plot_classes = ['F05', 'F04', top_absorber]
colors_map   = {'F05': 'tomato', 'F04': 'steelblue', top_absorber: 'seagreen'}

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()
for ax, feat in zip(axes, top6_feats):
    for cls in plot_classes:
        mask = train_df['Fault_Mode'] == cls
        vals = train_df.loc[mask, feat].dropna()
        ax.hist(vals, bins=15, alpha=0.55,
                label=f'{cls} (n={len(vals)})',
                color=colors_map.get(cls, 'gray'),
                edgecolor='white', linewidth=0.3)
    ax.set_title(feat, fontsize=9)
    ax.legend(fontsize=7)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
fig.suptitle(f'Feature distributions: F05 vs F04 vs {top_absorber}', fontsize=12)
plt.tight_layout()
plt.show()

### 6. F05 Individual Signals (Training Set)

In [ ]:
f05_train = train_df[train_df['Fault_Mode'] == 'F05'][['Experiment_Index', 'Serial_Number']]
print(f'F05 training experiments: {len(f05_train)}')
print(f05_train.to_string(index=False))

n = len(f05_train)
ncols = min(5, n)
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.5), squeeze=False)
for ax_i, (_, row) in enumerate(f05_train.iterrows()):
    ax = axes[ax_i // ncols][ax_i % ncols]
    sig = long_df[long_df['Experiment_Index'] == row['Experiment_Index']].sort_values('Sample_Index')
    ax.plot(sig['Sample_Index'], sig['Measurement'], color='tomato', linewidth=1)
    ax.set_title(f"Exp {row['Experiment_Index']}\n{row['Serial_Number']}", fontsize=8)
    ax.tick_params(labelsize=6)
for ax_i in range(n, nrows * ncols):
    axes[ax_i // ncols][ax_i % ncols].set_visible(False)
fig.suptitle('F05 individual signals (training set)', fontsize=12)
plt.tight_layout()
plt.show()